# Maps

How to write the interactive map as a standalone HTML file, from any set of fitted
ratings.

Contents:

1. [What the map needs](#needs)
2. [One USGS gage](#one-gage), with its position read from the site file
3. [Several gages on one map](#many-gages)
4. [Coordinates you supply yourself](#own-coords)
5. [The extras](#extras): a published rating, the cross-validation pane, where the file goes
6. [The MAGL shortcut](#magl)

Nothing here needs pagaia or the VPN: sections 1-5 run on public USGS data.

<a id="needs"></a>
## 1. What the map needs

`mapview.build_map` takes a list of
`SiteRating` - one per site - and writes one self-contained HTML file. A site is drawn
when it has both:

* `coords` - a `(latitude, longitude)` tuple, and
* at least one successful fit in `fits`.

A site missing either is skipped, and a call where every site is skipped raises
`ValueError` rather than writing an empty map.

`fits` is a list of
`FitResult`, which is what
`RatingModel.result` holds. So the route from the fitting interface to the map is:
fit with `lrc.compare(...)`, then hand `[model.result for model in comparison.models]`
to a `SiteRating` along with the coordinates. That is the helper the next cell defines,
and every section below goes through it.

In [ ]:
from pathlib import Path

import pandas as pd

import limnotech_rating_curves as lrc
from limnotech_rating_curves.core import SiteRating
from limnotech_rating_curves.data import usgs
from limnotech_rating_curves.view import mapview

#: Fitted at every site here. Any catalog keys work - lrc.models.catalog.all_keys().
MODELS = ["power_law", "spline"]

#: Where the HTML files below are written. build_map creates it.
OUTPUT = Path("maps_output")

In [ ]:
def site_for_map(comparison, *, sample_id, label, coords, source="sample", **fields):
    """Wrap a fitted RatingSet as the SiteRating the map draws.

    Parameters
    ----------
    comparison : lrc.RatingSet
        What lrc.compare(...) returned. Models that failed to fit are simply absent
        from it, so nothing has to be filtered here.
    sample_id : str
        Unique key for this site. The map's inspector is keyed on it.
    label : str
        The name shown in the tooltip and over the plot.
    coords : tuple
        (latitude, longitude) in decimal degrees.
    source : str
        Which marker style to draw it with: "usgs_gage", "colocated", "magl" or
        "pagaia" get their own colour and legend entry (see mapview.SOURCE_STYLE);
        anything else is drawn grey and labelled "sites".
    **fields
        Anything else SiteRating carries - gage, station, group, reference,
        extra_curves.

    Returns
    -------
    SiteRating
    """
    return SiteRating(
        sample_id=sample_id, source=source, label=label,
        sample=comparison.sample,
        fits=[model.result for model in comparison.models],
        coords=coords, **fields)

<a id="one-gage"></a>
## 2. One USGS gage

The field measurements and the coordinates come from the same site id. `gage_sample`
returns stage on the gage's own gage-height axis, so no datum conversion is needed, and
`coordinates` reads the decimal latitude and longitude out of the USGS site file. Both
are cached, so a second run does not refetch.

In [ ]:
GAGE = "04176356"      # Saline River at Austin Road near Bridgewater, MI

sample = usgs.gage_sample(GAGE)
usgs.site_info(GAGE).T

In [ ]:
# {site id: (latitude, longitude)}, omitting any site the file has no position for
usgs.coordinates(GAGE)

Fitting is the slow part: tens of seconds a site for these two models, and minutes once
the bdrc family is in the list, most of that PyTensor compiling its graph.
`method="advi"` is faster still and fine while laying a map out, but it produces no ELPD -
the map's tooltip ranks each site's models on predictive score, and with ADVI that ranking
falls back to R²log.

In [ ]:
comparison = lrc.compare(sample, models=MODELS)
comparison.metrics

In [ ]:
site = site_for_map(
    comparison,
    sample_id=f"usgs_gage:{GAGE}",
    label=f"USGS {GAGE} {usgs.site_info(GAGE).loc[GAGE, 'station_name']}",
    coords=usgs.coordinates(GAGE)[GAGE],
    source="usgs_gage",
    gage=GAGE)

mapview.build_map([site], OUTPUT / "one_gage.html", title=f"USGS {GAGE}")

That is the whole thing. The file is standalone - plotly.js is inlined - so open it in a
browser; it does not render usefully inside a notebook cell.

<a id="many-gages"></a>
## 3. Several gages on one map

The same call with a longer list. One fit per gage, so this is where the wall clock goes;
`lrc.fit_many` is the parallel version, but it returns saved ratings rather than the live
fits the map draws, so a map of many sites is normally built from `lrc.report` (section
6) or from a loop like this one.

In [ ]:
def usgs_site(gage, models=MODELS, coords=None, **fit_kwargs):
    """Fit one USGS gage and return it ready to map.

    `coords` defaults to the position in the USGS site file; pass a (lat, lon) tuple
    to override it, which is what to do for a gage whose site file has no position.
    """
    gage = usgs.normalize_site_id(gage)
    sample = usgs.gage_sample(gage)
    info = usgs.site_info(gage)
    name = str(info.loc[gage, "station_name"]) if gage in info.index else ""
    if coords is None:
        coords = usgs.coordinates(gage).get(gage)
    return site_for_map(
        lrc.compare(sample, models=models, **fit_kwargs),
        sample_id=f"usgs_gage:{gage}", label=f"USGS {gage} {name}".strip(),
        coords=coords, source="usgs_gage", gage=gage)

In [ ]:
GAGES = [GAGE, "04174500"]

sites = [usgs_site(gage) for gage in GAGES]
for mapped in sites:
    print(f"{mapped.sample_id}  {mapped.coords}  n={len(mapped.sample)}  "
          f"fitted {sum(fit.ok for fit in mapped.fits)} of {len(mapped.fits)}")

In [ ]:
mapview.build_map(sites, OUTPUT / "gages.html", title="two gages")

<a id="own-coords"></a>
## 4. Coordinates you supply yourself

Measurements that are not a USGS gage - a sensor, a hand-entered table, a CSV - carry no
site file, so the position is passed in. Everything else is the same: build a `Sample`,
compare, wrap, map.

In [ ]:
my_measurements = pd.DataFrame({
    "stage_ft": [1.42, 1.95, 2.60, 3.05, 3.88, 4.51, 5.60],
    "discharge_cfs": [3.1, 8.4, 21.0, 33.5, 72.0, 118.0, 214.0],
})

my_sample = lrc.Sample.of(my_measurements, site_id="MY-01", source="magl")

my_site = site_for_map(
    lrc.compare(my_sample, models=MODELS),
    sample_id="magl:MY-01",
    label="MY-01 (coordinates given by hand)",
    coords=(42.28, -83.74),           # decimal degrees, WGS84
    source="magl",
    station="MY-01_WL")

mapview.build_map([my_site], OUTPUT / "my_site.html", title="MY-01")

The two routes mix: a list can hold gages whose position was fetched and sites whose
position was typed, and each `source` gets its own colour and legend entry.

In [ ]:
mapview.build_map(sites + [my_site], OUTPUT / "gages_and_sensor.html",
                  title="gages and one sensor")

<a id="extras"></a>
## 5. The extras

**The published USGS rating.** A gage has one, and drawing it is the whole point of
mapping a gage - it is the closest thing to ground truth the site has. It goes on
`reference`, and it is only valid for a sample whose stage is on that gage's own
gage-height axis.

In [ ]:
with_reference = sites[0]          # the gage fitted in section 3

# None for a gage that publishes no rating; the map then simply draws none
with_reference.reference = usgs.published_reference(GAGE, with_reference.sample)

# drawn as a black dashed line, and scored on this site's own measurements
with_reference.reference["metrics"]

**The cross-validation pane.** Pass `cv_data`, keyed by `sample_id`, and the map grows a
second pane with a model selector and fold arrows. Each value is what
`CrossValidation.to_map_payload()` returns. Without it the pane and its controls are left
out and the score table takes the extra room.

In [ ]:
# three folds training on 70% of the measurements. The package's own default is a
# 90% holdout - deliberately harsh, and too little training data for the spline here.
result = lrc.cross_validate(with_reference.sample, models=MODELS, scheme="holdout",
                            holdout=0.3, n_splits=3)
cv_data = {with_reference.sample_id: result.to_map_payload()}

mapview.build_map([with_reference], OUTPUT / "cross_validated.html", cv_data=cv_data,
                  title=f"USGS {GAGE} with cross-validation")

**Where the file goes.** `build_map` returns the path it wrote and creates the parent
directory. Omit the path entirely and it writes `settings.MAP_HTML`, which is
`output/rating_curves_map.html` under the package unless `LRC_OUTPUT_DIR` moves it.

**Which site opens first.** `default_site=` takes a `sample_id`; the default is the first
site in the list.

In [ ]:
print(lrc.settings.MAP_HTML)
mapview.build_map(sites + [my_site], OUTPUT / "opens_on_my_site.html",
                  default_site=my_site.sample_id, title="opens on MY-01")

**Checking the figure without writing it.** `build_figure` returns
`(figure, script)` - the same assembly, for inspecting the traces in a test rather than
eyeballing a browser.

In [ ]:
figure, script = mapview.build_figure(sites, title="two gages")
pd.Series([trace.meta.get("kind") for trace in figure.data
           if trace.meta]).value_counts()

<a id="magl"></a>
## 6. The MAGL shortcut

For the MAGL network there is no assembly to do: `lrc.report(...)` selects the sites,
fetches stage and discharge, fits, scores, cross-validates and writes the map, and
reports where it went. It needs pagaia (and the VPN) for the station coordinates, which
is why the sections above build sites by hand instead.

```python
report = lrc.report(models=["power_law", "bdrc_gplm0"], sites=["SBR"],
                    cross_validate=True)
print(report.map_path)
```

The same run from the command line:

```
rating-curves --site SBR --models power_law bdrc_gplm0 --cv -v
```